# Experimento 3: Clustering Jerárquico

Este notebook implementa clustering jerárquico aglomerativo sobre la matriz normalizada de trayectorias académicas. Se evalúan diferentes configuraciones de número de clusters y método de enlace mediante métricas internas de validación.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

print("Entorno listo")

Entorno listo


In [2]:
ruta = Path("../data/processed/X_clustering.csv")

X = pd.read_csv(ruta)

X.head()

,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
0,-0.487231,0.516412,-0.402032,-0.464573,-0.711080,-0.596566,0.147024,0.486179,0.433139,0.673603,-0.961629
1,-0.487231,-0.420977,-0.487768,0.028994,0.942910,1.321249,0.383717,-1.921609,-2.051139,-0.304771,1.363608
2,-0.487231,-0.366903,-0.487768,0.028994,0.777511,1.129467,0.265371,-1.921609,-2.009536,-0.222381,1.221827
3,-1.416016,-2.899670,-1.387992,-1.385899,0.446713,0.745904,-1.825423,-1.754733,-1.108143,-2.636562,1.369857
4,0.441553,0.189696,0.555349,0.950319,-0.545681,-0.596566,-0.629817,0.831851,1.050932,0.599907,-0.535305


In [3]:
resultados_jerarquico = []


for linkage in ["ward", "complete", "average"]:

    for k in range(2, 11):

        modelo = AgglomerativeClustering(
            n_clusters=k,
            linkage=linkage
        )


        etiquetas = modelo.fit_predict(X)


        silhouette = silhouette_score(
            X,
            etiquetas
        )


        davies = davies_bouldin_score(
            X,
            etiquetas
        )


        resultados_jerarquico.append({
            "linkage": linkage,
            "k": k,
            "silhouette": silhouette,
            "davies_bouldin": davies
        })


resultados_jerarquico = pd.DataFrame(
    resultados_jerarquico
)


resultados_jerarquico

,linkage,k,silhouette,davies_bouldin
0,ward,2,0.497297,0.815482
1,ward,3,0.383058,0.972675
2,ward,4,0.417991,1.125397
3,ward,5,0.421194,0.940267
4,ward,6,0.430018,0.919620
5,ward,7,0.407951,0.957241
6,ward,8,0.339847,1.037705
7,ward,9,0.355756,0.949607
8,ward,10,0.337022,0.969030
9,complete,2,0.439305,1.023802


In [4]:
mejor_config = (
    resultados_jerarquico
    .sort_values(
        "silhouette",
        ascending=False
    )
    .iloc[0]
)

mejor_config

linkage            average
k                        2
silhouette        0.584301
davies_bouldin    0.297075
Name: 18, dtype: object

In [5]:
modelo_final = AgglomerativeClustering(
    n_clusters=int(mejor_config["k"]),
    linkage=mejor_config["linkage"]
)


clusters = modelo_final.fit_predict(X)

clusters

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0])

In [6]:
pd.Series(clusters).value_counts()

0    243
1      1
Name: count, dtype: int64

In [7]:
silhouette_final = silhouette_score(
    X,
    clusters
)

davies_final = davies_bouldin_score(
    X,
    clusters
)


print("Silhouette:", silhouette_final)
print("Davies-Bouldin:", davies_final)

Silhouette: 0.5843007034274259
Davies-Bouldin: 0.2970747974963721


In [8]:
resultado_jerarquico = pd.DataFrame({
    "modelo": ["Jerárquico"],
    "configuracion": [
        f'{mejor_config["linkage"]}, k={int(mejor_config["k"])}'
    ],
    "silhouette": [silhouette_final],
    "davies_bouldin": [davies_final]
})


resultado_jerarquico.to_csv(
    "../results/jerarquico/evaluacion_jerarquico.csv",
    index=False,
    encoding="utf-8-sig"
)


resultado_jerarquico

,modelo,configuracion,silhouette,davies_bouldin
0,Jerárquico,"average, k=2",0.584301,0.297075


In [9]:
df_original = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

resultado_clusters = df_original.copy()

resultado_clusters["cluster"] = clusters

resultado_clusters.groupby(
    "cluster"
).mean(numeric_only=True)

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
cluster,,,,,,,,,,,,
0,6.200999e+07,6.588477,13.225835,145.469136,47.263374,4.218107,3.024691,14.080462,11.562553,-2.517909,0.883549,2.949407
1,4.666558e+07,3.000000,4.638889,64.000000,12.000000,24.000000,24.000000,0.000000,0.000000,0.000000,0.333333,6.710984


In [11]:
modelo_ward = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward"
)

clusters_ward = modelo_ward.fit_predict(X)

pd.Series(clusters_ward).value_counts()

0    189
1     55
Name: count, dtype: int64

In [12]:
modelo_average3 = AgglomerativeClustering(
    n_clusters=3,
    linkage="average"
)

clusters_average3 = modelo_average3.fit_predict(X)

pd.Series(clusters_average3).value_counts()

0    238
2      5
1      1
Name: count, dtype: int64

In [13]:
silhouette_ward = silhouette_score(
    X,
    clusters_ward
)

davies_ward = davies_bouldin_score(
    X,
    clusters_ward
)

print("Silhouette:", silhouette_ward)
print("Davies-Bouldin:", davies_ward)

Silhouette: 0.49729661877713843
Davies-Bouldin: 0.8154819311369225


In [14]:
df_original = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

resultado_ward = df_original.copy()

resultado_ward["cluster"] = clusters_ward

resultado_ward.groupby(
    "cluster"
).mean(numeric_only=True)

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
cluster,,,,,,,,,,,,
0,6.218046e+07,7.412698,14.297740,163.359788,54.275132,2.206349,1.185185,14.144897,14.476311,0.331414,0.958375,2.052679
1,6.114520e+07,3.690909,9.386255,82.509091,22.527273,11.490909,9.727273,13.603030,1.339594,-12.263437,0.616419,6.099285


In [15]:
resultado_jerarquico = pd.DataFrame({
    "modelo": ["Jerárquico"],
    "configuracion": ["ward, k=2"],
    "silhouette": [silhouette_ward],
    "davies_bouldin": [davies_ward]
})

resultado_jerarquico.to_csv(
    "../results/jerarquico/evaluacion_jerarquico.csv",
    index=False,
    encoding="utf-8-sig"
)

In [16]:
caracterizacion_jerarquico = (
    resultado_ward
    .groupby("cluster")
    .mean(numeric_only=True)
)

caracterizacion_jerarquico.to_csv(
    "../results/jerarquico/caracterizacion_clusters.csv",
    encoding="utf-8-sig"
)